In [1]:
import itertools
import math
import cmath

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
import ffsim
from pyscf import gto, scf, cc
import json
from ffsim.linalg import givens_decomposition
from qiskit_ibm_runtime import  SamplerV2 as Sampler
from qiskit.circuit.library import XXPlusYYGate, RYGate, RZZGate, CPhaseGate

In [2]:
aer_sim = AerSimulator()
pm = generate_preset_pass_manager(optimization_level = 3, backend = aer_sim)

In [3]:
def generate_hchain_geometry(natoms: int, atomic_distance: float = 0.7) -> str:
    """Returns a linear Hydrogen chain geometry for use in PySCF molecule construction.
    
    Args:
        natoms: Number of Hydrogen atoms in the chain.
        atomic_distance: Equal spacing between Hydrogen atoms.
    """
    return "; ".join([f"H 0 0 {i * atomic_distance}" for i in range(natoms)])

In [5]:
geom = generate_hchain_geometry(natoms=40)
print(geom)

mol = gto.Mole(atom = geom, charge = 0, basis = 'sto3g')
mol.build()
mf = scf.RHF(mol)
mf.verbose = 0
mf.kernel()
cc_ = cc.CCSD(mf).run()
N = mol.nao_nr() * 2

H 0 0 0.0; H 0 0 0.7; H 0 0 1.4; H 0 0 2.0999999999999996; H 0 0 2.8; H 0 0 3.5; H 0 0 4.199999999999999; H 0 0 4.8999999999999995; H 0 0 5.6; H 0 0 6.3; H 0 0 7.0; H 0 0 7.699999999999999; H 0 0 8.399999999999999; H 0 0 9.1; H 0 0 9.799999999999999; H 0 0 10.5; H 0 0 11.2; H 0 0 11.899999999999999; H 0 0 12.6; H 0 0 13.299999999999999; H 0 0 14.0; H 0 0 14.7; H 0 0 15.399999999999999; H 0 0 16.099999999999998; H 0 0 16.799999999999997; H 0 0 17.5; H 0 0 18.2; H 0 0 18.9; H 0 0 19.599999999999998; H 0 0 20.299999999999997; H 0 0 21.0; H 0 0 21.7; H 0 0 22.4; H 0 0 23.099999999999998; H 0 0 23.799999999999997; H 0 0 24.5; H 0 0 25.2; H 0 0 25.9; H 0 0 26.599999999999998; H 0 0 27.299999999999997
E(CCSD) = -19.63931113769354  E_corr = -0.4108396472185855


In [6]:
def orbital_rotation(qc_lucj, orbital_rotation, N):
    givens_rotations, phase_shifts = givens_decomposition(orbital_rotation)
    for c, s, i, j in givens_rotations:
        # print(i, j)
        qc_lucj.append(XXPlusYYGate(theta = 2 * math.acos(c), beta = cmath.phase(s) - 0.5 * math.pi), [i, j])
        qc_lucj.append(XXPlusYYGate(theta = 2 * math.acos(c), beta = cmath.phase(s) - 0.5 * math.pi), [i + mol.nao_nr(), j + mol.nao_nr()])

    for i, phase_shift in enumerate(phase_shifts):
        qc_lucj.p(cmath.phase(phase_shift), i)
        qc_lucj.p(cmath.phase(phase_shift), i + mol.nao_nr())

    qc_lucj.barrier()   

In [7]:
def J_op(qc_lucj, diag_mat_aa, diag_mat_ab, diag_mat_bb, time, norb):
    for sigma, this_mat in enumerate([diag_mat_aa, diag_mat_bb]):
        if this_mat is None:
            print('______________________')
            print('_________NONE_________')
            print('______________________')
        if this_mat is not None:
            for i in range(norb):
                if this_mat[i, i]:
                    # print(i + sigma * norb)
                    qc_lucj.p(-0.5 * this_mat[i, i] * time, i + sigma * norb)
            for i, j in itertools.combinations(range(norb), 2):
                if this_mat[i, j]:
                    # print(i + sigma * norb, j + sigma * norb)
                    qc_lucj.cp(-this_mat[i, j] * time, i + sigma * norb, j + sigma * norb)
    # qc_lucj.barrier()
    if diag_mat_ab is not None:
        for i in range(norb):
            if diag_mat_ab[i, i]:
                qc_lucj.cp(-diag_mat_ab[i, i] * time, i, i + norb)
                # print(i,"%",i+norb)
        for i, j in itertools.combinations(range(norb), 2):
            
            if diag_mat_ab[i, j]:
                qc_lucj.cp(-diag_mat_ab[i, j] * time, i, j + norb)
                # print(i,"*",j+norb)
            if diag_mat_ab[j, i]:
                qc_lucj.cp(-diag_mat_ab[j, i] * time, j, i + norb)
                # print(j,"$",i+norb)

    qc_lucj.barrier()

In [8]:
qc_lucj = QuantumCircuit(N)
t1, t2 = cc_.t1, cc_.t2
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(t2 = cc_.t2, t1 = cc_.t1)
# qc_lucj.x([0, 1, 2, 7, 8, 9])
orbital_rotations = ucj_op.orbital_rotations
diag_mats = ucj_op.diag_coulomb_mats
final_orbital_rotation = ucj_op.final_orbital_rotation
for i in range(mol.nelectron // 2):
    qc_lucj.x(i)
    qc_lucj.x(i + mol.nao_nr())
# qc_lucj.barrier()
for orb_rot, (diag_mat_aa, diag_mat_ab) in zip(orbital_rotations, diag_mats):
    # print(linalg.givens_decomposition(orbital_rotations[l]), l)
    orbital_rotation(qc_lucj, orb_rot.T.conj(), N)
    # qc_lucj.barrier()
    J_op(qc_lucj, diag_mat_aa, diag_mat_ab, diag_mat_aa, time = -1, norb = mol.nao_nr())
    # qc_lucj.barrier()
    orbital_rotation(qc_lucj, orb_rot, N)
if ucj_op.final_orbital_rotation is not None:
    orbital_rotation(qc_lucj, ucj_op.final_orbital_rotation, N)

# qc_lucj.draw(fold=-1)

/Users/ryan/prof/work/lucj/envlucj/lib/python3.10/site-packages/ffsim/linalg/predicates.py:83: RuntimeWarning: divide by zero encountered in matmul
  return m == n and np.allclose(mat @ mat.T.conj(), np.eye(m), rtol=rtol, atol=atol)
/Users/ryan/prof/work/lucj/envlucj/lib/python3.10/site-packages/ffsim/linalg/predicates.py:83: RuntimeWarning: overflow encountered in matmul
  return m == n and np.allclose(mat @ mat.T.conj(), np.eye(m), rtol=rtol, atol=atol)
/Users/ryan/prof/work/lucj/envlucj/lib/python3.10/site-packages/ffsim/linalg/predicates.py:83: RuntimeWarning: invalid value encountered in matmul
  return m == n and np.allclose(mat @ mat.T.conj(), np.eye(m), rtol=rtol, atol=atol)


In [9]:
qc_lucj.count_ops()

OrderedDict([('cp', 2528000),
             ('xx_plus_yy', 2482696),
             ('p', 192080),
             ('barrier', 2401),
             ('x', 40)])

In [10]:
qc_lucj.measure_all()
backend = AerSimulator()
sampler = Sampler(mode=backend)
pm=generate_preset_pass_manager(backend=backend,optimization_level=3)
isa_circuit=pm.run(qc_lucj)
#decomposed_circuit = circuit.decompose()
pub_job = sampler.run([isa_circuit], shots=100_000)

pub_result=pub_job.result()[0]
print(pub_result.data.meas.get_counts())
l1 = pub_result.data.meas.get_counts()

KeyboardInterrupt: 

In [ ]:
# print("orbital_rotations =",orbital_rotations)
# print("diag_mats =",diag_mats)
# print("final_orbital_rotations =",final_orbital_rotation)

In [11]:
def dump_lucj_to_json(ucj_op, orbital_rotations, diag_mats,
                      time, norb, nelectron, filepath):
    """Serialize a full LUCJ ansatz to JSON for the Julia side to consume."""

    def cx_to_pair(z):
        z = complex(z)
        return [float(z.real), float(z.imag)]

    def decompose_unitary(U):
        givens_rots, phase_shifts = givens_decomposition(U)    # <-- direct call
        rots_serialized = []
        for g in givens_rots:
            c, s, i, j = g
            rots_serialized.append({
                "c":    float(c.real) if hasattr(c, "real") else float(c),
                "s_re": float(complex(s).real),
                "s_im": float(complex(s).imag),
                "i":    int(i),
                "j":    int(j),
            })
        phases_serialized = [cx_to_pair(p) for p in phase_shifts]
        return {"givens": rots_serialized, "phase_shifts": phases_serialized}

    layers = []
    for orb_rot, (diag_mat_aa, diag_mat_ab) in zip(orbital_rotations, diag_mats):
        layers.append({
            "fwd":         decompose_unitary(orb_rot.T.conj()),
            "inv":         decompose_unitary(orb_rot),
            "diag_mat_aa": [[float(x) for x in row] for row in diag_mat_aa],
            "diag_mat_ab": [[float(x) for x in row] for row in diag_mat_ab],
        })

    final_serialized = None
    if ucj_op.final_orbital_rotation is not None:
        final_serialized = decompose_unitary(ucj_op.final_orbital_rotation)

    payload = {
        "norb":      norb,
        "time":      float(time),
        "nelectron": int(nelectron),
        "layers":    layers,
        "final":     final_serialized,
    }

    with open(filepath, "w") as f:
        json.dump(payload, f, indent=2)

    print(f"Wrote LUCJ data to {filepath}")
    print(f"  norb={norb}, time={time}, nelectron={nelectron}, "
          f"{len(layers)} layer(s), final_rotation="
          f"{'yes' if final_serialized is not None else 'no'}")

# Usage
dump_lucj_to_json(
    ucj_op, orbital_rotations, diag_mats,
    time=-1.0,
    norb=mol.nao_nr(),
    nelectron=mol.nelectron,
    filepath="lucj_params.json",
)

Wrote LUCJ data to lucj_params.json
  norb=40, time=-1.0, nelectron=40, 800 layer(s), final_rotation=yes


In [ ]:
from pyscf import tools
tools.fcidump.from_scf(mf, 'h20.fcidump')

In [ ]:
print("mf.e_tot:", mf.e_tot)                  # total HF energy (electronic + nuclear)
print("mol.energy_nuc():", mol.energy_nuc())  # nuclear repulsion
print("electronic HF:", mf.e_tot - mol.energy_nuc())